# Lab 1: Anatomy of a Decision

**Series**: Foundation Layer | **Time**: ~45 min

Understand why prompt structure shifts tool selection probability.

```
P(tool | prompt) -- descriptions, ordering, format, temperature all matter
```

## Learning Objectives

By the end of this lab, you will be able to:
- Explain how prompt structure affects tool selection probability
- Design controlled experiments to measure agent decision quality
- Identify the impact of temperature, description quality, and format constraints
- Apply these findings to build more reliable scientific agents

## Why This Matters for Scientists

In scientific computing, **reproducibility** is non-negotiable. If your agent selects different tools for the same query depending on how you phrase the prompt, your results aren't reproducible.

This lab treats the agent's decision-making process as a **scientific experiment**: you'll control variables (prompt structure, temperature, format), measure outcomes (tool selection frequency), and draw conclusions. You're applying the scientific method to AI engineering.

> **Key insight**: "The prompt IS the architecture." — Small changes in prompt structure cause large changes in agent behavior. Understanding this is the difference between a demo and a production system.

## Prerequisites & NVIDIA SDK Connection

| Requirement | Details |
|------------|---------|
| Completed | Lab 0 (Agent Prototype) |
| Concepts | System prompts, tool selection |

**NVIDIA Connection**: When using **NVIDIA NIM** with Nemotron models, you may observe different tool selection patterns than with OpenAI models. This is expected — different models have different decision boundaries. The experiments in this lab help you calibrate for your chosen model, whether it's `gpt-4o-mini` or `nvidia/llama-3.3-nemotron-super-49b-v1.5`.

### Install Dependencies

In [ ]:
!pip install -q openai

### Connect to LLM
Same setup as Lab 0 — connects to either OpenAI or NVIDIA NIM.

> **Why NIM?** Data privacy, faster inference, open models. See Lab 0 for the full comparison.

In [ ]:
import os
from getpass import getpass
from openai import OpenAI
use_nim = os.environ.get("USE_NIM","").lower() in ("1","true","yes") or "NIM_API_KEY" in os.environ
if use_nim:
    if "NIM_API_KEY" not in os.environ: os.environ["NIM_API_KEY"]=getpass("NVIDIA NIM API key: ")
    client=OpenAI(base_url="https://integrate.api.nvidia.com/v1",api_key=os.environ["NIM_API_KEY"])
    MODEL=os.environ.get("NIM_MODEL","nvidia/llama-3.3-nemotron-super-49b-v1.5")
else:
    if "OPENAI_API_KEY" not in os.environ: os.environ["OPENAI_API_KEY"]=getpass("OpenAI API key: ")
    client=OpenAI()
    MODEL="gpt-4o-mini"
print(f"Using model: {MODEL}")

### Import Utilities
We import `re` for regex parsing and `Counter` for counting tool selection frequencies across multiple runs.

In [ ]:
import re
from collections import Counter

### Set Up Tools and Prompt Builder
We define the same two tools from Lab 0, plus a flexible `build_prompt()` function that lets us experiment with different prompt structures.

In [ ]:
TOOLS={"search_literature":"Search scientific databases for papers by keyword. Use when user wants to find research papers.","summarize_findings":"Summarize research results into key findings. Use when user wants a concise overview."}

def build_prompt(tools,header=None):
    h=header or "You are a scientific research assistant. Available tools:"
    lines=[h,""]
    for i,(name,desc) in enumerate(tools.items(),1): lines.append(f"  {i}. {name} -- {desc}")
    lines.extend(["","Choose exactly one tool. Reply with ONE line:","TOOL: <tool_name>","No other text."])
    return "\n".join(lines)

def parse_tool(text):
    import re; m=re.search(r"TOOL:\s*(\S+)",text.strip(),re.IGNORECASE)
    return m.group(1) if m else None

### Single Tool Selection
This function sends one message to the LLM and returns which tool it selected. We'll use it to run controlled experiments.

In [ ]:
def select_tool(msg,tools=None,temperature=0.0,header=None):
    if tools is None: tools=TOOLS
    r=client.chat.completions.create(model=MODEL,temperature=temperature,max_tokens=30,messages=[{"role":"system","content":build_prompt(tools,header)},{"role":"user","content":msg}])
    text=(r.choices[0].message.content or "").strip()
    return {"raw":text,"parsed_tool":parse_tool(text)}

### Batch Testing Helper
To measure tool selection *probability*, we need to run the same query multiple times and count results. `run_batch()` does this and displays a frequency histogram.

In [ ]:
def run_batch(msg,tools=None,n=8,temperature=0.0,label=""):
    results=[select_tool(msg,tools=tools,temperature=temperature) for _ in range(n)]
    freq=Counter(r["parsed_tool"] for r in results)
    print(f"\n--- {label or msg[:55]} (n={n}, temp={temperature}) ---")
    for tool,count in freq.most_common():
        pct=count/n*100; print(f"  {str(tool):<25} {count}/{n} ({pct:.0f}%) {chr(9608)*int(pct/10)}")

r=select_tool("Find papers about mRNA vaccine efficacy.")
print("Baseline:",r["parsed_tool"],"|",r["raw"])

## Concept: Tool Selection as a Probability Distribution

When an LLM decides which tool to call, it's not a deterministic lookup — it's a **probability distribution** over all possible tools.

```
P(tool | user_message, system_prompt, temperature)
```

Three factors shift this distribution:

| Factor | Effect | Lab Experiment |
|--------|--------|---------------|
| **Description quality** | Specific descriptions → sharper distribution | Experiment 1 |
| **Temperature** | Higher temp → flatter distribution (more random) | Experiment 2 |
| **Format contract** | Missing format → unparseable output | Experiment 3 |

We'll measure this by running the **same query multiple times** and counting which tool gets selected. This is a Monte Carlo estimate of the tool selection probability.

## Experiment 1: Description Quality

### Run Experiment 1: Description Quality
We compare **specific** tool descriptions (from Lab 0) vs. **vague** descriptions ('Do a search.', 'Write a summary.'). Watch how the quality of descriptions affects which tool gets selected.

In [ ]:
VAGUE={"search_literature":"Do a search.","summarize_findings":"Write a summary."}
q="Find recent clinical studies on metformin and type 2 diabetes."
run_batch(q,tools=TOOLS,n=5,temperature=0.3,label="Specific")
run_batch(q,tools=VAGUE,n=5,temperature=0.3,label="Vague")

<details>
<summary>Expected output (click to expand)</summary>

```
--- Specific (n=5, temp=0.3) ---
  search_literature         5/5 (100%) ██████████

--- Vague (n=5, temp=0.3) ---
  search_literature         3/5 (60%) ██████
  summarize_findings        2/5 (40%) ████
```

> **Note**: Your output may differ slightly due to LLM non-determinism. The structure and key information should be similar.
</details>

## Concept: Temperature and Determinism

The `temperature` parameter controls the randomness of the LLM's output:

| Temperature | Behavior | Use Case |
|-------------|----------|----------|
| **0.0** | Deterministic — same input → same output | Tool selection, structured extraction |
| **0.3** | Slightly creative — small variations | Scientific writing with some variety |
| **0.7** | Moderate creativity | Brainstorming, hypothesis generation |
| **1.2+** | High creativity — unpredictable | Creative tasks (NOT for tool selection) |

> **Rule of thumb for scientific agents**: Use `temperature=0.0` for any decision that must be reproducible (tool selection, data extraction). Use `temperature=0.3-0.7` only for creative tasks like hypothesis generation.

## Experiment 2: Temperature Sweep

### Run Experiment 2: Temperature Sweep
We send the same **ambiguous** query ('I need information about cancer research') at different temperatures. At temperature=0, the model always picks the same tool. As temperature increases, choices become more random.

In [ ]:
ambiguous="I need information about cancer research."
for t in [0.0,0.3,0.7,1.2]: run_batch(ambiguous,n=8,temperature=t,label=f"temp={t}")

<details>
<summary>Expected output (click to expand)</summary>

```
--- temp=0.0 (n=8, temp=0.0) ---
  search_literature         8/8 (100%) ██████████

--- temp=0.3 (n=8, temp=0.3) ---
  search_literature         7/8 (88%) ████████

--- temp=0.7 (n=8, temp=0.7) ---
  search_literature         5/8 (62%) ██████
  summarize_findings        3/8 (38%) ███

--- temp=1.2 (n=8, temp=1.2) ---
  search_literature         4/8 (50%) █████
  summarize_findings        3/8 (38%) ███
  None                      1/8 (12%) █
```

> **Note**: Your output may differ slightly due to LLM non-determinism. The structure and key information should be similar.
</details>

## Concept: The Format Contract

The format contract (`TOOL: <name>`) serves as a **communication protocol** between the LLM and your parser. Without it:

- The LLM might describe which tool to use in natural language ("I would recommend using the search tool...")
- Your regex parser can't extract a clean tool name
- The agent fails silently — no tool gets called

This is why Lab 2 introduces **OpenAI function calling** with Pydantic schemas: it replaces the fragile regex-based format contract with a robust, typed interface.

## Experiment 3: Format Drift

Remove the `TOOL: <name>` format instruction.

### Run Experiment 3: Format Drift
What happens when we remove the `TOOL: <name>` format instruction? The LLM might respond in free text that our parser can't handle. This shows why format contracts matter.

In [ ]:
def build_no_format(tools):
    lines=["You are a helpful scientific assistant. Available tools:",""]
    for i,(name,desc) in enumerate(tools.items(),1): lines.append(f"  {i}. {name} -- {desc}")
    lines.append("
Please help the user."); return "
".join(lines)
q="Find papers about CRISPR gene editing."
print("=== WITH format ===")
for _ in range(3):
    r=select_tool(q,temperature=0.0); print(f"  parsed={r[chr(39)]parsed_tool{chr(39)]!r:<20} raw={r[chr(39)]raw{chr(39)]}")
print("
=== WITHOUT format ===")
for _ in range(3):
    r2=client.chat.completions.create(model=MODEL,temperature=0.0,max_tokens=80,messages=[{"role":"system","content":build_no_format(TOOLS)},{"role":"user","content":q}])
    text=(r2.choices[0].message.content or "").strip(); print(f"  parsed={parse_tool(text)!r:<20} raw={text[:80]}")

<details>
<summary>Expected output (click to expand)</summary>

```
=== WITH format ===
  parsed='search_literature' raw=TOOL: search_literature
  parsed='search_literature' raw=TOOL: search_literature
  parsed='search_literature' raw=TOOL: search_literature

=== WITHOUT format ===
  parsed=None               raw=I can help you find papers about CRISPR gene editing! Let me
  parsed=None               raw=I can help you find papers about CRISPR gene editing! Let me
  parsed=None               raw=I can help you find papers about CRISPR gene editing! Let me
```

> **Note**: Your output may differ slightly due to LLM non-determinism. The structure and key information should be similar.
</details>

## Reflection Questions

1. **You found that vague tool descriptions reduce accuracy.** How would you write descriptions for tools in your own scientific domain? What makes a description "specific enough"?
2. **Temperature=0 is best for tool selection.** But what about an agent that needs to *generate novel hypotheses*? How would you handle both needs in a single agent?
3. **The format contract is brittle.** What would a more robust alternative look like? (Hint: Lab 2 answers this with Pydantic schemas.)

## Three Takeaways

1. **Description quality** determines correct routing.
2. **Temperature=0** for reliable tool selection.
3. **Format contract** () is required.

**Next**: Lab 2 -- Pydantic schemas for typed tool arguments.

---
*Agentic AI Science Playbook -- Foundation Layer, Lab 1.*